# 12 Similarity Model Comparison

## Purpose

This notebook compares similarity-scaleup outputs from notebook 8 across any set of model runs.

For the current default run, I am comparing three `manual` tokenizer model states:
- pretrained MLM checkpoint
- classification fine-tuned from MLM initialization
- classification fine-tuned from random initialization

The goal is to answer the professor's question more directly: do the embeddings before and after fine-tuning produce different similarity distributions and different similarity clouds?

This notebook does not load model weights or compute embeddings. It only reads CSV outputs from notebook 8, so it should run fine without a GPU.

It also saves a simple HTML gallery report with clickable plots, cartoons, labels, side-by-side neighbor clouds, and Venn-style cloud-overlap diagrams so the comparison is easier to inspect visually.


## Setup note

Same Colab pattern as the other notebooks:
- code lives in GitHub
- notebook 8 outputs live in Drive
- this notebook pulls the repo and imports helper functions from `src/`

If a file is missing, that usually means the matching notebook 8 run has not been run yet.


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')


## User settings

This is the main cell to edit if I rerun this for another tokenizer or another group of models later.

The important idea is that `RUN_SPECS` is the comparison recipe. As long as each folder has the standard notebook 8 output files, the rest of the notebook should work the same way.


In [ ]:
# ==============================================================================
# 1. IMPORT HELPERS AND DEFINE THE COMPARISON RUN
# ==============================================================================
import importlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

import src.similarity_model_comparison as similarity_model_comparison
importlib.reload(similarity_model_comparison)

from src.similarity_model_comparison import build_similarity_model_comparison

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'
OUTPUT_RUN_LABEL = 'live_extended'

SIMILARITY_SCALEUP_ROOT = DRIVE_ROOT / 'results' / 'similarity_scaleup'
CLASSIFICATION_SCALEUP_ROOT = SIMILARITY_SCALEUP_ROOT / 'classification'
CLASSIFICATION_EVALUATION_ROOT = DRIVE_ROOT / 'results' / 'classification_evaluation'

# These are the three notebook-8 output folders I want to compare.
# The model_id values are short because they become table columns and plot labels.
RUN_SPECS = [
    {
        'model_id': 'pretrained_mlm',
        'model_label': 'Pretrained MLM',
        'run_dir': SIMILARITY_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / OUTPUT_RUN_LABEL,
    },
    {
        'model_id': 'classification_mlm_init',
        'model_label': 'Classifier, MLM init',
        'run_dir': CLASSIFICATION_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / 'cls_lr2e-5_ep100_bs16_mlm' / OUTPUT_RUN_LABEL,
        'classification_evaluation_dir': CLASSIFICATION_EVALUATION_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / 'cls_lr2e-5_ep100_bs16_mlm',
    },
    {
        'model_id': 'classification_random_init',
        'model_label': 'Classifier, random init',
        'run_dir': CLASSIFICATION_SCALEUP_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / 'cls_lr2e-5_ep100_bs16_randominit' / OUTPUT_RUN_LABEL,
        'classification_evaluation_dir': CLASSIFICATION_EVALUATION_ROOT / TOKENIZER_FAMILY / EXPERIMENT_NAME / 'cls_lr2e-5_ep100_bs16_randominit',
    },
]

# This label table lets me ask whether a cloud is semantically cleaner, not just
# whether cosine scores changed. Missing labels are counted separately.
LABEL_TABLE_PATH = DRIVE_ROOT / 'results' / 'classification_prep' / 'labeled_glycans_with_split.csv'

OUTPUT_DIR = (
    DRIVE_ROOT
    / 'results'
    / 'similarity_model_comparison'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
    / 'pretrain_vs_classifier_mlm_vs_randominit'
)

# These thresholds are recomputed from the ranked similarity tables.
# 0.90 matches the cloud view I have been looking at most closely.
CLOUD_THRESHOLDS = [0.90, 0.85, 0.80]
TOP_K_NEIGHBORS = 25

# These settings control the visual HTML gallery report.
# The HTML report is meant for quick human inspection, not for heavy statistics.
# HTML_CLOUD_THRESHOLD controls the neighbor clouds used in the Venn diagrams.
# EMBED_HTML_IMAGES=True makes the downloaded HTML work as one standalone file.
HTML_CLOUD_THRESHOLD = 0.90
HTML_TOP_N_NEIGHBORS = 8
HTML_REPORT_TITLE = f'{TOKENIZER_FAMILY} similarity model comparison'
EMBED_HTML_IMAGES = True

def stringify_config_paths(config_dict):
    # The saved config is just for record-keeping, so Path objects become strings.
    return {
        key: str(value) if isinstance(value, Path) else value
        for key, value in config_dict.items()
    }

comparison_config = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'experiment_name': EXPERIMENT_NAME,
    'output_run_label': OUTPUT_RUN_LABEL,
    'label_table_path': str(LABEL_TABLE_PATH),
    'output_dir': str(OUTPUT_DIR),
    'cloud_thresholds': CLOUD_THRESHOLDS,
    'top_k_neighbors': TOP_K_NEIGHBORS,
    'html_cloud_threshold': HTML_CLOUD_THRESHOLD,
    'html_top_n_neighbors': HTML_TOP_N_NEIGHBORS,
    'html_report_title': HTML_REPORT_TITLE,
    'embed_html_images': EMBED_HTML_IMAGES,
    'run_specs': [
        stringify_config_paths(spec)
        for spec in RUN_SPECS
    ],
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'similarity_model_comparison_config.json').write_text(
    json.dumps(comparison_config, indent=2),
    encoding='utf-8',
)

print(f'Comparison output dir: {OUTPUT_DIR}')
for spec in RUN_SPECS:
    print(f'- {spec["model_label"]}: {spec["run_dir"]} | exists={Path(spec["run_dir"]).exists()}')
    if 'classification_evaluation_dir' in spec:
        print(f'  evaluation: {spec["classification_evaluation_dir"]} | exists={Path(spec["classification_evaluation_dir"]).exists()}')
print(f'Label table: {LABEL_TABLE_PATH} | exists={LABEL_TABLE_PATH.exists()}')


## Run the comparison

This cell reads the notebook 8 CSVs, builds the comparison tables, and saves the plots.

No GPU is needed here because the embeddings were already created by notebook 8.


In [ ]:
# ==============================================================================
# 2. BUILD COMPARISON TABLES AND PLOTS
# ==============================================================================
comparison_outputs = build_similarity_model_comparison(
    run_specs=RUN_SPECS,
    label_table_path=LABEL_TABLE_PATH,
    output_dir=OUTPUT_DIR,
    cloud_thresholds=CLOUD_THRESHOLDS,
    top_k_neighbors=TOP_K_NEIGHBORS,
    html_cloud_threshold=HTML_CLOUD_THRESHOLD,
    html_top_n_neighbors=HTML_TOP_N_NEIGHBORS,
    report_title=HTML_REPORT_TITLE,
    embed_html_images=EMBED_HTML_IMAGES,
)

comparison_tables = comparison_outputs['tables']

print('Saved comparison tables:')
for name, path in comparison_outputs['table_paths'].items():
    print(f'- {name}: {path}')

print('Saved comparison plots:')
for name, path in comparison_outputs['plot_paths'].items():
    print(f'- {name}: {path}')

print(f'Manifest: {comparison_outputs["manifest_path"]}')
print(f'HTML report: {comparison_outputs["plot_paths"]["html_report_path"]}')

# Colab can display this link inline. This one report is the main shareable file.
display(HTML(
    f'<p><a href="{comparison_outputs["plot_paths"]["html_report_path"]}" target="_blank">Open full HTML comparison report</a></p>'
))


## Review the whole-space distribution

This is the broadest check: did the overall embedding-space similarity distribution move across the three model states?


In [ ]:
# ==============================================================================
# 3. REVIEW ALL-VS-ALL SUMMARY
# ==============================================================================
all_vs_all_df = comparison_tables['all_vs_all_model_comparison']
summary_cols = ['model_id', 'model_label', 'num_glycans', 'count', 'mean', 'median', 'std_dev', 'q05', 'q25', 'q75', 'q95']
display(all_vs_all_df[summary_cols])
display(Image(filename=comparison_outputs['plot_paths']['all_vs_all_plot']))


## Review the query-specific clouds

This is closer to the professor's question about similarity clouds. I am checking whether the same query glycans have different similarity distributions under the three models.


In [ ]:
# ==============================================================================
# 4. REVIEW SPECIFIC-VS-ALL QUERY SUMMARIES
# ==============================================================================
specific_df = comparison_tables['specific_vs_all_model_comparison']
query_cols = ['model_id', 'model_label', 'query_accession', 'mean', 'median', 'std_dev', 'q05', 'q25', 'q75', 'q95', 'max']
display(specific_df[query_cols].sort_values(['query_accession', 'model_id']))
display(Image(filename=comparison_outputs['plot_paths']['specific_vs_all_plot']))


## Compare cloud sizes and cloud overlap

Cloud size answers: how many structures are above a similarity threshold?

Cloud overlap answers: are the models putting the same structures into the cloud, or different ones?


In [ ]:
# ==============================================================================
# 5. REVIEW THRESHOLD CLOUD SIZE AND OVERLAP
# ==============================================================================
threshold_size_df = comparison_tables['threshold_cloud_size_model_comparison']
cloud_overlap_df = comparison_tables['threshold_cloud_overlap_model_comparison']

display(threshold_size_df.sort_values(['query_accession', 'threshold', 'model_id']))
display(cloud_overlap_df.sort_values(['query_accession', 'threshold', 'model_a', 'model_b']).head(30))


## Compare top-neighbor stability

This checks whether the top neighbors are stable across model states. Low overlap means the cloud changed in a very concrete way.


In [ ]:
# ==============================================================================
# 6. REVIEW TOP-NEIGHBOR OVERLAP
# ==============================================================================
top_neighbor_overlap_df = comparison_tables['top_neighbor_overlap_model_comparison']
display(top_neighbor_overlap_df.sort_values(['query_accession', 'model_a', 'model_b']))


## Review label overlap inside the clouds

This is the most semantic check in this notebook. For each query cloud, I count how many labeled neighbors share at least one subtype label with the query.

Important note: missing labels are counted separately, not as wrong labels.


In [ ]:
# ==============================================================================
# 7. REVIEW LABEL OVERLAP IN THRESHOLD CLOUDS
# ==============================================================================
cloud_label_df = comparison_tables.get('cloud_label_overlap_model_comparison')

if cloud_label_df is None:
    print('No label-overlap table was created because LABEL_TABLE_PATH was not set.')
else:
    review_cols = [
        'model_id',
        'model_label',
        'query_accession',
        'threshold',
        'query_labels_json',
        'cloud_size',
        'labeled_neighbors',
        'neighbors_without_labels',
        'exact_label_set_matches',
        'any_label_overlap',
        'no_label_overlap',
        'exact_label_set_match_rate',
        'any_label_overlap_rate',
    ]
    display(cloud_label_df[review_cols].sort_values(['query_accession', 'threshold', 'model_id']))


## Final note

The first thing I would report from this notebook is whether the three all-vs-all distributions separate, then whether the query clouds change size/composition, then whether label overlap gets better or worse after classification fine-tuning.

For the professor's question, the key comparison is pretrained MLM vs classifier fine-tuned from MLM init. The random-init classifier is useful as a sanity-check baseline.
